[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hashirama21/neoplasia-detection/blob/main/RARE2026.ipynb)

# RARE2026 — Barrett's Neoplasia Detection

**Task:** Binary classification (neoplasia vs. NDBE) on endoscopy images.  
**Metric:** PPV @ 90% Recall — bootstrap-simulated at 1% clinical prevalence.  
**Strategy:** DINOv2 ViT-B/14 + GastroNet-5M weights · LoRA (rank=8) · Asymmetric Loss · Isotonic calibration.

---

### Notebook structure
| Section | Content |
|---|---|
| 1 | BONSAI dataset — download, EDA, patient-aware splits |
| 2 | EVC Barretts FullSet — structure, masks, texture analysis |
| 3 | Dataset integration — merge train + enrich val_calibration |
| 4 | Repository setup, dependencies, GastroNet weights |
| 5 | Training — 5 independent seeds (true ensemble) |
| 6 | Calibration + bootstrap evaluation |
| 7 | Inference demo |

In [ ]:
# ── Runtime constants — edit only these paths ──────────────────────────────
REPO_DIR    = '/content/rare26'
DATA_DIR    = '/content/data'
EVC_DIR     = '/content/EVC_Barretts_FullSet'
WEIGHTS_DIR = '/content/rare26/weights'
OUTPUT_DIR  = '/content/outputs'

import os, torch
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(WEIGHTS_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
if device == 'cuda':
    name = torch.cuda.get_device_name(0)
    mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU : {name}  ({mem:.1f} GB)')
else:
    print('No GPU — activate: Runtime → Change runtime type → GPU')

---
## Section 1 — BONSAI Dataset (Official RARE26 Training Data)

3 095 images from 2 endoscopy centres (center_1 / center_2).  
158 neoplasia (neo) · 2 937 NDBE.  
Images are stored by `center/class/` with hash filenames — no visible patient ID in the filename.

In [ ]:
import os, zipfile, gdown

BONSAI_FILE_ID  = '1Hs9O6Gckq3CUuPMN5Symq5roQahreYnx'
BONSAI_ZIP      = '/content/bonsai_dataset.zip'
BONSAI_EXTRACT  = '/content/bonsai_raw'

if not os.path.exists(BONSAI_EXTRACT):
    print('Downloading BONSAI dataset...')
    gdown.download(f'https://drive.google.com/uc?id={BONSAI_FILE_ID}', BONSAI_ZIP, quiet=False)
    os.makedirs(BONSAI_EXTRACT, exist_ok=True)
    with zipfile.ZipFile(BONSAI_ZIP, 'r') as z:
        z.extractall(BONSAI_EXTRACT)
    print('Extraction complete.')
else:
    print('BONSAI data already extracted.')

# Quick structure check
for root, dirs, files in os.walk(BONSAI_EXTRACT):
    imgs = [f for f in files if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    if imgs:
        rel = os.path.relpath(root, BONSAI_EXTRACT)
        print(f'  {rel:<40} {len(imgs):>5} images')

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from pathlib import Path

EXTS = {'.jpg', '.jpeg', '.png'}

rows = []
for root, _, files in os.walk(BONSAI_EXTRACT):
    imgs = [f for f in files if Path(f).suffix.lower() in EXTS]
    if not imgs: continue
    parts = Path(root).relative_to(BONSAI_EXTRACT).parts
    center  = parts[0] if len(parts) > 0 else 'unknown'
    cls_raw = parts[1] if len(parts) > 1 else 'unknown'
    for f in imgs:
        rows.append({'image_path': os.path.join(root, f), 'center': center, 'class_raw': cls_raw})

bonsai_df = pd.DataFrame(rows)
NEG_CLASS = 'ndbe'
bonsai_df['label'] = (bonsai_df['class_raw'] != NEG_CLASS).astype(int)

print(f'Total images : {len(bonsai_df)}')
print(bonsai_df.groupby(['center', 'class_raw'])['label'].count().to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=bonsai_df, x='center', hue='class_raw', ax=axes[0])
axes[0].set_title('Images per centre')
sns.countplot(data=bonsai_df, x='class_raw', ax=axes[1], palette=['steelblue', 'tomato'])
axes[1].set_title('Class balance  (pos=neo)')
plt.tight_layout()
plt.show()

In [ ]:
rng = np.random.default_rng(42)
sample = bonsai_df.sample(min(200, len(bonsai_df)), random_state=42)

size_rows = []
for _, row in sample.iterrows():
    try:
        with Image.open(row['image_path']) as img:
            w, h = img.size
            size_rows.append({'class_raw': row['class_raw'], 'width': w, 'height': h})
    except Exception:
        continue

sizes_df = pd.DataFrame(size_rows)
print('Image dimension statistics:')
print(sizes_df.groupby('class_raw')[['width', 'height']].describe().T.to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.scatterplot(data=sizes_df, x='width', y='height', hue='class_raw', alpha=0.5, ax=axes[0])
axes[0].set_title('Resolution scatter')
for col, color, ax in [('width', 'royalblue', axes[1]), ('height', 'tomato', axes[1])]:
    sizes_df[col].plot(kind='hist', bins=20, alpha=0.5, color=color, label=col, ax=ax)
axes[1].legend(); axes[1].set_title('Width / Height distributions')
plt.tight_layout(); plt.show()

print(f"\nMedian size: {sizes_df['width'].median():.0f} x {sizes_df['height'].median():.0f} px")
print("-> 392px crop is well-matched (28x28 patches of 14px for DINOv2)")

### 1.3 — Patient-aware train/val split

BONSAI filenames are hashes — no visible patient ID. We apply `StratifiedShuffleSplit` on images here,  
but **EVC data will use `GroupShuffleSplit` by `patient_id`** (filenames: `patXX_imY_DIAGNOSIS`).  
When BONSAI patient metadata is available, replace `StratifiedShuffleSplit` with `GroupShuffleSplit` here.

In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit, GroupShuffleSplit

# Save full metadata CSV
bonsai_df[['image_path', 'label']].to_csv(f'{DATA_DIR}/bonsai_all.csv', index=False)

# Step 1: 85/15 train/val split (stratified by label)
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
train_idx, val_idx = next(sss1.split(np.zeros(len(bonsai_df)), bonsai_df['label'].values))
train_df = bonsai_df.iloc[train_idx][['image_path', 'label']].reset_index(drop=True)
val_df   = bonsai_df.iloc[val_idx][['image_path', 'label']].reset_index(drop=True)

# Step 2: val → val_selection (70%) + val_calibration (30%)
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=42)
sel_idx, cal_idx = next(sss2.split(np.zeros(len(val_df)), val_df['label'].values))
val_sel_df = val_df.iloc[sel_idx].reset_index(drop=True)
val_cal_df = val_df.iloc[cal_idx].reset_index(drop=True)

train_df.to_csv(f'{DATA_DIR}/train.csv',              index=False)
val_df.to_csv(f'{DATA_DIR}/val.csv',                  index=False)
val_sel_df.to_csv(f'{DATA_DIR}/val_selection.csv',    index=False)
val_cal_df.to_csv(f'{DATA_DIR}/val_calibration.csv',  index=False)

print('BONSAI splits written:')
for name, df in [('train', train_df), ('val_selection', val_sel_df), ('val_calibration', val_cal_df)]:
    print(f'  {name:<18}: {len(df):5d} images  pos={df["label"].sum():3d}  neg={(df["label"]==0).sum():4d}')

---
## Section 2 — EVC Barretts FullSet

External dataset from a previous TU/e challenge. **Used to enrich val_calibration** (target ≥ 50 positives)  
and expand the training set.

**Structure:** `patXX_imY_DIAGNOSIS.{png,bmp}` — clear patient ID in filename.  
**Labels:** `ACHD` = neoplasia (label=1) · `NDBT` = normal (label=0).  
**Annotations:** per-pixel expert masks in BMP format (5 annotators).  
**Risk note:** ACHD lesions here have ~17% image coverage on average — visible lesions.  
RARE26 test targets subtle lesions at <1% prevalence. EVC improves recall on easy cases,  
not necessarily on subtle ones. Verify after Open Phase submission.

**Expected path:** `EVC_DIR = /content/EVC_Barretts_FullSet`  
Upload the ZIP to Colab or mount Drive, then extract to that path.

In [ ]:
import os
from pathlib import Path

EVC_EXPECTED = Path(EVC_DIR)

if EVC_EXPECTED.exists():
    n_imgs = sum(1 for p in EVC_EXPECTED.rglob('*') if p.suffix.lower() in {'.png', '.bmp', '.jpg'})
    print(f'EVC found at {EVC_DIR}  ({n_imgs} files total)')
    EVC_AVAILABLE = True
else:
    print(f'EVC not found at {EVC_DIR}.')
    print('To extract from a ZIP already in Colab:')
    print("  import zipfile")
    print("  with zipfile.ZipFile('/content/EVC_Barretts_FullSet.zip') as z:")
    print(f"      z.extractall('{EVC_DIR}')")
    EVC_AVAILABLE = False

In [ ]:
# Only runs if EVC data is present
if not EVC_AVAILABLE:
    print('Skip — EVC not available. Run the extraction cell above first.')
else:
    import re
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns
    from pathlib import Path

    EXTS_IMG  = {'.png', '.jpg', '.jpeg'}
    EXTS_MASK = {'.bmp'}

    file_data = []
    for root, _, files in os.walk(EVC_DIR):
        for f in files:
            path = os.path.join(root, f)
            ext  = Path(f).suffix.lower()
            stem = Path(f).stem
            parts = stem.split('_')
            if len(parts) >= 3:
                file_data.append({
                    'filepath': path,
                    'filename': f,
                    'patient_id': parts[0],
                    'image_id':   parts[1],
                    'diagnosis':  parts[2],
                    'file_type':  'mask' if ext in EXTS_MASK else 'image',
                    'extension':  ext,
                })

    evc_meta = pd.DataFrame(file_data)
    evc_imgs = evc_meta[evc_meta['file_type'] == 'image'].copy()

    print(f'EVC total files parsed : {len(evc_meta)}')
    print(f'  Image files          : {len(evc_imgs)}')
    print(f'  Mask files (BMP)     : {(evc_meta["file_type"]=="mask").sum()}')
    print()
    print('Diagnosis distribution:')
    print(evc_imgs['diagnosis'].value_counts().to_string())
    print()
    print(f'Unique patients : {evc_imgs["patient_id"].nunique()}')
    print(evc_imgs.groupby("diagnosis")["patient_id"].nunique().rename("unique patients").to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.countplot(data=evc_imgs, x='diagnosis', palette=['steelblue', 'tomato'], ax=axes[0])
    axes[0].set_title('EVC — images per diagnosis')
    pat_counts = evc_imgs.groupby(['patient_id', 'diagnosis']).size().reset_index(name='count')
    sns.histplot(data=pat_counts, x='count', hue='diagnosis', bins=20, ax=axes[1])
    axes[1].set_title('Images per patient')
    plt.tight_layout()
    plt.show()

In [ ]:
if not EVC_AVAILABLE:
    print('Skip.')
else:
    import random
    from PIL import Image
    import matplotlib.pyplot as plt

    print('Sample images — ACHD (neoplasia) vs NDBT (normal):')
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))

    for row_idx, diag in enumerate(['ACHD', 'NDBT']):
        samples = evc_imgs[evc_imgs['diagnosis'] == diag].sample(4, random_state=42)
        for col_idx, (_, meta) in enumerate(samples.iterrows()):
            ax = axes[row_idx][col_idx]
            try:
                img = Image.open(meta['filepath']).convert('RGB')
                ax.imshow(img)
            except Exception as e:
                ax.text(0.5, 0.5, str(e), ha='center', va='center', fontsize=8)
            ax.set_title(f"{diag} | {meta['patient_id']}", fontsize=9)
            ax.axis('off')

    plt.suptitle('EVC — visual comparison ACHD vs NDBT', fontsize=13)
    plt.tight_layout()
    plt.show()

In [ ]:
if not EVC_AVAILABLE:
    print('Skip.')
else:
    import cv2
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    import seaborn as sns

    evc_masks = evc_meta[evc_meta['file_type'] == 'mask'].copy()

    mask_stats = []
    for _, row in evc_masks.sample(min(150, len(evc_masks)), random_state=42).iterrows():
        mask = cv2.imread(row['filepath'], cv2.IMREAD_GRAYSCALE)
        if mask is not None:
            coverage = float((mask > 0).mean() * 100)
            mask_stats.append({'diagnosis': row['diagnosis'], 'coverage_pct': coverage})

    df_masks = pd.DataFrame(mask_stats)

    print('Expert annotation coverage (% of image area annotated as lesion):')
    print(df_masks.groupby('diagnosis')['coverage_pct'].describe().round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.violinplot(data=df_masks, x='diagnosis', y='coverage_pct', ax=axes[0],
                   palette=['steelblue', 'tomato'], inner='box')
    axes[0].set_title('Lesion coverage (%) — expert masks')
    axes[0].set_ylabel('Coverage (%)')

    achd_cov = df_masks[df_masks['diagnosis'] == 'ACHD']['coverage_pct']
    axes[1].hist(achd_cov, bins=20, color='tomato', edgecolor='white')
    axes[1].axvline(achd_cov.median(), color='black', linestyle='--',
                    label=f'Median {achd_cov.median():.1f}%')
    axes[1].set_title('ACHD lesion coverage distribution')
    axes[1].legend()
    plt.tight_layout()
    plt.show()

    print(f"\nKey finding: ACHD median coverage = {achd_cov.median():.1f}%")
    print("RARE26 test targets subtle lesions (<1% prevalence) — EVC improves recall")
    print("on visible lesions but generalisation to subtle cases requires test-set verification.")

In [ ]:
if not EVC_AVAILABLE:
    print('Skip.')
else:
    from skimage.feature import local_binary_pattern
    from skimage.color import rgb2gray
    import cv2, numpy as np, pandas as pd
    import matplotlib.pyplot as plt, seaborn as sns

    evc_png = evc_imgs[evc_imgs['extension'] == '.png']
    sampled = evc_png.groupby('diagnosis').sample(min(50, len(evc_png)//2), random_state=42)

    R, N_PTS = 3, 24
    texture_rows = []
    for _, row in sampled.iterrows():
        img = cv2.imread(row['filepath'])
        if img is None: continue
        gray = rgb2gray(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        lbp  = local_binary_pattern(gray, N_PTS, R, method='uniform')
        texture_rows.append({
            'diagnosis':        row['diagnosis'],
            'texture_variance': float(np.var(lbp)),
            'brightness_std':   float(np.std(gray)),
        })

    df_tex = pd.DataFrame(texture_rows)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.kdeplot(data=df_tex, x='texture_variance', hue='diagnosis', fill=True, ax=axes[0])
    axes[0].set_title('Texture variance (LBP)')
    sns.boxplot(data=df_tex, x='diagnosis', y='brightness_std', ax=axes[1],
                palette=['steelblue', 'tomato'])
    axes[1].set_title('Brightness heterogeneity')
    plt.tight_layout()
    plt.show()

    print('Note: DINOv2 GastroNet-5M captures these texture patterns at all scales')
    print('— manual LBP/GLCM features add noise not signal for this backbone.')

---
## Section 3 — Dataset Integration

Merge EVC into the BONSAI training set using `scripts/prepare_evc.py`.  
Objective: **≥ 50 positives in val_calibration** so bootstrap recall is continuous and interpretable.  
EVC ACHD patients are split by `patient_id` — no patient appears in both train and calibration.

In [ ]:
if not EVC_AVAILABLE:
    print('EVC not available — skipping integration.')
    print('Training will use BONSAI-only splits. val_calibration has ~7 positives.')
    print('Metrics will have high variance. Integrate EVC before the final run.')
    TRAIN_CSV  = f'{DATA_DIR}/train.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration.csv'
else:
    !python {REPO_DIR}/scripts/prepare_evc.py \
        --evc_dir {EVC_DIR} \
        --train_csv {DATA_DIR}/train.csv \
        --val_cal_csv {DATA_DIR}/val_calibration.csv \
        --out_dir {DATA_DIR} \
        --target_cal_pos 50 \
        --seed 42

    import pandas as pd
    TRAIN_CSV   = f'{DATA_DIR}/train_merged.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration_enriched.csv'

    train_m  = pd.read_csv(TRAIN_CSV)
    val_cal  = pd.read_csv(VAL_CAL_CSV)
    val_sel  = pd.read_csv(f'{DATA_DIR}/val_selection.csv')

    print('\nFinal dataset summary:')
    for name, df in [('train_merged', train_m), ('val_selection', val_sel), ('val_cal_enriched', val_cal)]:
        print(f'  {name:<20}: {len(df):5d} images  pos={df["label"].sum():3d}  neg={(df["label"]==0).sum():4d}')

    if val_cal['label'].sum() < 35:
        print('\n⚠ val_calibration has fewer than 35 positives.')
        print('  Bootstrap recall will be discrete and metrics unreliable.')
        print('  Add more EVC ACHD patients or reduce --target_cal_pos.')
    else:
        print(f'\n✓ val_calibration has {val_cal["label"].sum()} positives — metrics will be interpretable.')

In [ ]:
# Rebuild splits with patient-level grouping if EVC filenames allow it
# (BONSAI uses hash names; EVC uses patXX names — this is handled inside prepare_evc.py)
# Run rebuild_splits.py only if you need to regenerate existing BONSAI splits:

# !python {REPO_DIR}/scripts/rebuild_splits.py \
#     --val_csv {DATA_DIR}/val.csv \
#     --sel_csv {DATA_DIR}/val_selection.csv \
#     --cal_csv {DATA_DIR}/val_calibration.csv \
#     --sel_ratio 0.70 --seed 42

print('Splits are ready. Proceeding to Section 4.')

---
## Section 4 — Repository Setup, Dependencies, and GastroNet Weights

In [ ]:
import os, sys
from pathlib import Path

REPO_DIR = '/content/rare26'

if not Path(REPO_DIR).exists():
    !git clone https://github.com/hashirama21/neoplasia-detection.git {REPO_DIR}
else:
    !git -C {REPO_DIR} pull origin main

sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Core dependencies
%pip install -r {REPO_DIR}/requirements.txt -q

# LoRA adapter — required for parameter-efficient backbone fine-tuning
%pip install peft -q

# Verify peft is importable
try:
    import peft
    print(f'peft {peft.__version__} — LoRA will be active')
except ImportError:
    print('WARNING: peft import failed — LoRA will be disabled (differential LR fallback)')

In [ ]:
from pathlib import Path
import torch, timm

WEIGHTS_PATH = Path(WEIGHTS_DIR) / 'dinov2_gastronet5m.pth'
Path(WEIGHTS_DIR).mkdir(parents=True, exist_ok=True)

if WEIGHTS_PATH.exists():
    print(f'Weights already present: {WEIGHTS_PATH}')
else:
    # Strategy 1: GastroNet-5M from HuggingFace (endoscopy domain pretraining)
    try:
        from huggingface_hub import hf_hub_download
        print('Trying GastroNet-5M from HuggingFace (BONS-AI-TUE-AMC/GastroNetDinov2)...')
        hf_hub_download(
            repo_id='BONS-AI-TUE-AMC/GastroNetDinov2',
            filename='dinov2_gastronet5m.pth',
            local_dir=WEIGHTS_DIR,
        )
        print('GastroNet-5M downloaded.')
    except Exception as e:
        print(f'GastroNet-5M unavailable ({e}). Falling back to ImageNet DINOv2...')
        # Strategy 2: ImageNet DINOv2 via timm
        backbone = timm.create_model('vit_base_patch14_dinov2.lvd142m', pretrained=True, num_classes=0)
        torch.save(backbone.state_dict(), WEIGHTS_PATH)
        print('ImageNet DINOv2 weights saved (domain shift vs GastroNet-5M expected).')

# Verify
if WEIGHTS_PATH.exists():
    ckpt   = torch.load(WEIGHTS_PATH, map_location='cpu', weights_only=True)
    state  = ckpt if not any(k in ckpt for k in ('model_state','state_dict','model')) else \
             (ckpt.get('model_state') or ckpt.get('state_dict') or ckpt.get('model') or ckpt)
    n_par  = sum(v.numel() for v in state.values() if isinstance(v, torch.Tensor))
    sample = list(state.keys())[:4]
    print(f'\nCheckpoint: {WEIGHTS_PATH.stat().st_size/1e6:.1f} MB | {n_par/1e6:.1f}M params')
    print(f'Keys sample: {sample}')
    print('\nWeights ready — training will load via configs/model/dinov2_gastronet.yaml')
else:
    print('No weights found — training will use random initialization.')

---
## Section 5 — Training

**5 independent runs** with different seeds → true ensemble (not 5 checkpoints of the same run).  
Each run saves its top-3 checkpoints to `outputs/seed_X/checkpoints/`.  
Ensemble aggregation is by mean logits after calibration.

| Config | Value |
|---|---|
| Epochs | 30 |
| Image size | 392 px (28×28 DINOv2 patches) |
| Backbone LR | 1e-5 (LoRA only) |
| Head LR | 1e-3 |
| Loss | AsymmetricLoss (γ⁻=4, γ⁺=1, clip=0.05) |
| Scheduler | CosineAnnealing, T_max=epochs |
| Seeds | 42, 123, 456, 789, 1337 |

**Estimated time per run:** ~7 min (L4) · ~15 min (T4) · ~4 min (A100)

In [ ]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# Use merged CSVs if EVC was integrated, else BONSAI-only
try:
    TRAIN_CSV
except NameError:
    TRAIN_CSV   = f'{DATA_DIR}/train.csv'
    VAL_CAL_CSV = f'{DATA_DIR}/val_calibration.csv'

SEEDS = [42, 123, 456, 789, 1337]

for seed in SEEDS:
    print(f'\n{"="*60}')
    print(f'Run  seed={seed}')
    print(f'{"="*60}')

    seed_output = f'{OUTPUT_DIR}/seed_{seed}'

    !python {REPO_DIR}/scripts/train.py \
        project.output_dir={seed_output} \
        project.seed={seed} \
        paths.data_dir={DATA_DIR} \
        paths.weights_dir={WEIGHTS_DIR} \
        device={device} \
        num_workers=2 \
        pin_memory=True \
        training.epochs=30 \
        training.cross_validation.enabled=false \
        training.loss.gamma_neg=4 \
        data.image_size=392 \
        data.augmentation.val.center_crop=392 \
        data.train_csv={TRAIN_CSV} \
        data.val_calibration_csv={VAL_CAL_CSV} \
        data.batch_size=12 \
        data.oversample_factor=3.0 \
        data.pos_weight_factor=18.6 \
        model.backbone.img_size=392 \
        model.lora.enabled=true

print('\nAll 5 training runs complete.')

In [ ]:
import json, glob
import pandas as pd
from pathlib import Path

rows = []
for seed in SEEDS:
    seed_output = f'{OUTPUT_DIR}/seed_{seed}'
    cal_file = Path(seed_output) / 'results' / 'calibration_results.json'
    if cal_file.exists():
        with open(cal_file) as f:
            r = json.load(f)
        rows.append({
            'seed':          seed,
            'median_ppv':    r.get('median_ppv', float('nan')),
            'median_recall': r.get('median_recall', float('nan')),
            'std_ppv':       r.get('std_ppv', float('nan')),
            'threshold':     r.get('optimal_threshold', float('nan')),
        })
    else:
        rows.append({'seed': seed, 'median_ppv': None, 'note': 'no results yet'})

df_runs = pd.DataFrame(rows)
print('Training summary (calibration set results):')
from IPython.display import display
display(df_runs)

---
## Section 6 — Calibration & Bootstrap Evaluation

Evaluation is on **val_selection** (never on val_calibration which was used for threshold).  
Three stability criteria before Docker submission:
- PPV std < 0.15 → calibration is stable
- Median recall ≥ 0.88 → recall constraint satisfied with margin
- P10 PPV > 0.30 → correct behaviour even in worst bootstrap samples

In [ ]:
# Run ensemble evaluation using all 5 seed checkpoints
# evaluate.py should be configured to load checkpoints from all seed directories

!python {REPO_DIR}/scripts/evaluate.py \
    project.output_dir={OUTPUT_DIR} \
    paths.data_dir={DATA_DIR} \
    paths.weights_dir={WEIGHTS_DIR} \
    device={device} \
    num_workers=2 \
    pin_memory=True

In [ ]:
import json, pandas as pd
from pathlib import Path
from IPython.display import display

eval_path = Path(OUTPUT_DIR) / 'results' / 'evaluation_results.json'
if not eval_path.exists():
    # Try first seed's results as fallback
    eval_path = Path(OUTPUT_DIR) / 'seed_42' / 'results' / 'evaluation_results.json'

if eval_path.exists():
    with open(eval_path) as f:
        ev = json.load(f)

    metrics = {
        'Median PPV@90Recall': ev.get('median_ppv', ev.get('median_ppv_at_90recall', 'N/A')),
        'Mean PPV':            ev.get('mean_ppv', 'N/A'),
        'Std PPV':             ev.get('std_ppv', 'N/A'),
        'P10 PPV':             ev.get('p10_ppv', 'N/A'),
        'P90 PPV':             ev.get('p90_ppv', 'N/A'),
        'Median Recall':       ev.get('median_recall', 'N/A'),
        'Threshold':           ev.get('optimal_threshold', ev.get('threshold_used', 'N/A')),
    }

    df_ev = pd.DataFrame(metrics.items(), columns=['Metric', 'Value'])
    df_ev['Value'] = df_ev['Value'].apply(lambda x: f'{x:.4f}' if isinstance(x, float) else str(x))
    display(df_ev.set_index('Metric'))

    # Stability check
    std   = ev.get('std_ppv', 999)
    rec   = ev.get('median_recall', 0)
    p10   = ev.get('p10_ppv', 0)
    med   = ev.get('median_ppv', ev.get('median_ppv_at_90recall', 0))

    checks = [
        (std  < 0.15, f'PPV std < 0.15        : {std:.4f}'),
        (rec  >= 0.88, f'Median recall ≥ 0.88  : {rec:.4f}'),
        (p10  > 0.30,  f'P10 PPV > 0.30        : {p10:.4f}'),
    ]

    print('\nStability criteria for Docker submission:')
    all_pass = True
    for passed, label in checks:
        mark = '✓' if passed else '✗'
        print(f'  {mark} {label}')
        if not passed: all_pass = False

    if all_pass:
        print('\n✓ All criteria met — ready for Open Development Phase submission.')
    else:
        print('\n✗ Not all criteria met. Investigate before submitting.')
else:
    print(f'No evaluation results found at {eval_path}')
    print('Run the evaluation cell above first.')

Run 1 — E001–E020 results (val_calibration, BONSAI-only, no LoRA):

Metric                 Value
---------------------  ------
Median PPV@90Recall    1.0000
Mean PPV               0.9890
Std PPV                0.1043
P10 PPV                1.0000
P90 PPV                1.0000
Median Recall          0.6667
Threshold              0.5010

Note: val_selection has only ~17 positives (3 unique after 70% split).
These metrics are not interpretable at this sample size.
EVC integration (Section 3) is required for reliable evaluation.


---
## Section 7 — Inference Demo

Run the best checkpoint on 4 random images from the dataset.

In [ ]:
import glob, json, random, sys
import numpy as np
import matplotlib.pyplot as plt
import torch, torchvision.transforms as T
from pathlib import Path
from PIL import Image

sys.path.insert(0, REPO_DIR)
from src.models.rare26_model import Rare26Model
from src.calibration.calibrator import IsotonicCalibrator
from omegaconf import OmegaConf

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Find best checkpoint across all seeds
all_ckpts = sorted(
    glob.glob(f'{OUTPUT_DIR}/**/checkpoints/*.pt', recursive=True)
    + glob.glob(f'{OUTPUT_DIR}/checkpoints/*.pt'),
    key=lambda p: float(Path(p).stem.rsplit('_', 1)[-1]),
    reverse=True,
)

if not all_ckpts:
    print('No checkpoint found — run Section 5 first.')
else:
    best_ckpt = all_ckpts[0]
    print(f'Loading checkpoint: {Path(best_ckpt).name}')

    model_cfg = OmegaConf.load(f'{REPO_DIR}/configs/model/dinov2_gastronet.yaml')
    OmegaConf.update(model_cfg, 'checkpoint_path', str(Path(WEIGHTS_DIR) / 'dinov2_gastronet5m.pth'))
    OmegaConf.update(model_cfg, 'backbone.img_size', 392)
    model = Rare26Model(model_cfg).to(device)

    ckpt = torch.load(best_ckpt, map_location=device, weights_only=True)
    model.load_state_dict(ckpt['model_state'])
    model.eval()

    # Calibrator and threshold
    cal_pkl  = Path(OUTPUT_DIR) / 'results' / 'isotonic_calibrator.pkl'
    cal_json = Path(OUTPUT_DIR) / 'results' / 'calibration_results.json'
    if not cal_pkl.exists():  # try seed_42
        cal_pkl  = Path(OUTPUT_DIR) / 'seed_42' / 'results' / 'isotonic_calibrator.pkl'
        cal_json = Path(OUTPUT_DIR) / 'seed_42' / 'results' / 'calibration_results.json'

    calibrator = None
    threshold  = 0.5
    if cal_pkl.exists():
        calibrator = IsotonicCalibrator()
        calibrator.load(str(cal_pkl))
    if cal_json.exists():
        with open(cal_json) as f:
            threshold = json.load(f).get('optimal_threshold', 0.5)

    print(f'Calibrator: {"loaded" if calibrator else "not found"}  |  Threshold: {threshold:.4f}')

    # Inference transform — must match training (392px)
    transform = T.Compose([
        T.Resize(448, interpolation=T.InterpolationMode.BICUBIC),
        T.CenterCrop(392),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])

    # Sample images from BONSAI
    all_imgs = (
        glob.glob(f'{BONSAI_EXTRACT}/**/*.jpg', recursive=True) +
        glob.glob(f'{BONSAI_EXTRACT}/**/*.png', recursive=True)
    )
    samples = random.sample(all_imgs, min(4, len(all_imgs)))

    fig, axes = plt.subplots(1, len(samples), figsize=(16, 4))
    if len(samples) == 1: axes = [axes]

    for ax, img_path in zip(axes, samples):
        img = Image.open(img_path).convert('RGB')
        with torch.no_grad():
            tensor  = transform(img).unsqueeze(0).to(device)
            logit   = model(tensor).squeeze().item()
        raw_prob = 1.0 / (1.0 + np.exp(-logit))
        cal_prob = float(calibrator.transform(np.array([raw_prob]))[0]) if calibrator else raw_prob
        pred     = cal_prob >= threshold

        # Ground truth from folder name
        parent   = Path(img_path).parent.name.lower()
        true_cls = 'neo' if parent not in ('ndbe', 'ndbt') else 'ndbe'

        color = 'red' if pred else 'green'
        ax.imshow(img)
        ax.set_title(
            f'True: {true_cls}\nProb: {cal_prob:.3f} → {"NEOPLASIA" if pred else "NDBE"}',
            color=color, fontsize=9,
        )
        ax.axis('off')

    plt.suptitle(f'Inference demo — calibrated threshold: {threshold:.3f}', fontsize=12)
    plt.tight_layout()
    plt.show()